# CatBoost Model – Advanced Modeling (Day 7)

## Objective
Train and tune a CatBoost classifier to predict customer churn using the
approved feature set, and evaluate performance using ROC-AUC.

## Imports & Setup

In [33]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils.config import PROCESSED_DATA_PATH

In [34]:
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


## Load Processed Dataset

In [35]:
df = pd.read_csv(PROCESSED_DATA_PATH / "telco_customer_churn_selected_features.csv")

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

## Train / Validation Split


In [36]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

## Baseline CatBoost Model


In [37]:
baseline_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

baseline_model.fit(X_train, y_train)

y_train_proba = baseline_model.predict_proba(X_train)[:, 1]
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

print(f"Baseline ROC AUC (Train): {roc_auc_score(y_train, y_train_proba):.4f}")
print(f"Baseline ROC AUC: {roc_auc_score(y_val, y_val_proba):.4f}")

Baseline ROC AUC (Train): 0.9199
Baseline ROC AUC: 0.8338


- A baseline CatBoost model was trained using default hyperparameters.
The model shows a reasonable generalization gap, which is expected for
boosting-based models and motivates further hyperparameter tuning.

## Hyperparameter Tuning


In [38]:
model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

param_dist = {
    "iterations": [300, 500, 700],
    "learning_rate": [0.03, 0.05],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [3, 5, 10],
    "subsample": [0.7, 0.85, 1.0],
}

In [39]:
model.randomized_search(
    param_dist,
    X=X_train,
    y=y_train,
    cv=3,
    n_iter=60,
    partition_random_seed=42,
    verbose=False,
)


bestTest = 0.8442203492
bestIteration = 309

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8420931166
bestIteration = 359

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8441360626
bestIteration = 532

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8427975115
bestIteration = 501

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8427072045
bestIteration = 294

Metric AUC is not calculated on train by default. To calculate this metric on train, add hints=skip_train~false to metric parameters.

bestTest = 0.8406803131
bestIteration = 177

Metric AUC is not calculated on train by default. To c

{'params': {'subsample': 0.7,
  'depth': 4,
  'learning_rate': 0.05,
  'l2_leaf_reg': 3,
  'iterations': 500},
 'cv_results': defaultdict(list,
             {'iterations': [0,
               1,
               2,
               3,
               4,
               5,
               6,
               7,
               8,
               9,
               10,
               11,
               12,
               13,
               14,
               15,
               16,
               17,
               18,
               19,
               20,
               21,
               22,
               23,
               24,
               25,
               26,
               27,
               28,
               29,
               30,
               31,
               32,
               33,
               34,
               35,
               36,
               37,
               38,
               39,
               40,
               41,
               42,
               43,
               4

## Best Hyperparameters with randomized search
  * subsample: 0.7
  * depth: 4
  * learning_rate: 0.05
  * l2_leaf_reg: 3
  * iterations: 500

## Validation Performance


In [40]:
y_train_proba = model.predict_proba(X_train)[:, 1]
y_val_proba = model.predict_proba(X_val)[:, 1]

print(f"tuned model ROC AUC (Train): {roc_auc_score(y_train, y_train_proba):.4f}")
print(f"tured model ROC AUC: {roc_auc_score(y_val, y_val_proba):.4f}")

tuned model ROC AUC (Train): 0.8888
tured model ROC AUC: 0.8380


## Manual Configuration Check
- The model exhibited mild variance, which motivated testing slightly stronger
  regularization and fewer boosting iterations to further improve generalization.

In [41]:
tuned_params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 4,
    'l2_leaf_reg': 3,
    'subsample': 0.7
  }

config_1 = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 4,
    "l2_leaf_reg": 7, #
    "subsample": 0.7
}

config_2 = {
    'iterations': 400, #
    'learning_rate': 0.05,
    'depth': 4,
    'l2_leaf_reg': 3,
    'subsample': 0.7
  }

config_3 = {
    'iterations': 400, #
    'learning_rate': 0.05,
    'depth': 4,
    'l2_leaf_reg': 7, #
    'subsample': 0.7
  }



def evaluate_config(params):
    model = CatBoostClassifier(
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=42,
        logging_level="Silent",
        allow_writing_files=False,
        **params
    )
    model.fit(X_train, y_train)

    val_auc = roc_auc_score(
        y_val,
        model.predict_proba(X_val)[:, 1]
    )
    train_auc = roc_auc_score(
        y_train,
        model.predict_proba(X_train)[:, 1]
    )

    print(f"ROC AUC (Train): {train_auc*100:.2f}%")
    print(f"ROC AUC: {val_auc*100:.2f}%")
    return train_auc, val_auc

In [42]:
evaluate_config(tuned_params)
evaluate_config(config_1)
evaluate_config(config_2)
evaluate_config(config_3)


ROC AUC (Train): 88.88%
ROC AUC: 83.80%
ROC AUC (Train): 88.71%
ROC AUC: 83.86%
ROC AUC (Train): 88.21%
ROC AUC: 83.91%
ROC AUC (Train): 88.03%
ROC AUC: 84.00%


(0.8802522865539556, 0.8400307422046552)

### Observations

Reducing model complexity through stronger regularization and fewer iterations
led to a small decrease in training performance but a consistent improvement in
validation ROC-AUC, indicating better generalization.

## Best Hyperparameters

The following configuration was selected as the final CatBoost model based on
validation performance and stability:

- subsample: 0.7  
- depth: 4  
- learning_rate: 0.05  
- l2_leaf_reg: 7  
- iterations: 400  

## Conclusion

CatBoost was evaluated as an advanced modeling approach using the finalized
feature set. Initial baseline training demonstrated strong performance but
exhibited mild variance. Hyperparameter tuning via randomized search identified
a robust parameter region, confirming the model’s ability to generalize well.

A limited manual configuration check was subsequently performed to validate and
refine the tuning results. Controlled adjustments to the number of boosting
iterations and regularization strength resulted in a small but consistent
improvement in validation ROC-AUC, indicating improved generalization while
maintaining model simplicity.

Based on these findings, the final CatBoost configuration was selected from the
manually refined candidates, achieving a balanced bias–variance trade-off and
representing the strongest overall model observed during experimentation.